# 02 Managed vs External Tables

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Know the difference between <b>managed</b> and <b>external</b> tables in Unity Catalog: who controls the storage location, what happens on <code>DROP</code>, and when to use each. Learn how storage credentials and external locations connect Unity Catalog to cloud storage, how to inspect a table's type and location, and how to recover a dropped managed table with <code>UNDROP</code>.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Choosing between managed and external tables decides what happens to your data when someone drops a table, which optimizations Databricks can apply automatically, and how other tools can access the files. It's a classic exam topic: "What happens to the data when you drop an external table?"
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Managed vs external</b><br>
| | Managed table | External table |<br>|---|---|---|<br>| Storage location | Chosen by Unity Catalog (managed storage of the metastore, catalog or schema) | A path you specify with <code>LOCATION</code>, inside an <b>external location</b> |<br>| <code>DROP TABLE</code> | Removes metadata, and the data files are deleted (after a retention period; <code>UNDROP</code> possible for 7 days) | Removes metadata only. <b>Files stay</b> in cloud storage |<br>| Format | Delta (or Iceberg) | Delta, Parquet, CSV, JSON, Avro, ORC, text |<br>| Automatic optimizations | Predictive optimization, automatic liquid clustering, faster metadata | Fewer |<br>| Typical use | The default for almost everything | Data also written or read by other systems; data that must outlive the table definition |<br><br>
How external storage is secured:<br><code>`</code><br>Storage credential  (cloud identity: Azure managed identity / AWS IAM role / GCP service account)<br>   └── External location  (credential + path, e.g. abfss://raw@acct.dfs.core.windows.net/sales)<br>          └── External tables and external volumes under that path<br><code>`</code><br>Admins create credentials and locations once. Users get privileges like <code>CREATE EXTERNAL TABLE</code> and <code>READ FILES</code> on the location, never cloud keys.
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A data team stored all Silver tables as external tables "to be safe". When a developer cleaned up dev tables with <code>DROP TABLE</code>, the files stayed behind and storage costs kept growing, and when they recreated a table at the same path, old files mixed with new data. Moving to managed tables gave them automatic <code>OPTIMIZE</code> and <code>VACUUM</code> through predictive optimization, and <code>UNDROP</code> for mistakes. They kept external tables only for a landing zone that a partner system writes to directly.
</div>

## Syntax

```sql
-- managed: no LOCATION
CREATE TABLE workspace.governance.orders (id BIGINT, amount DOUBLE);

-- external: LOCATION inside an external location you can use
CREATE TABLE prod.raw.partner_orders (id BIGINT, amount DOUBLE)
LOCATION 'abfss://landing@myaccount.dfs.core.windows.net/partner_orders';

DESCRIBE TABLE EXTENDED t;      -- Type: MANAGED / EXTERNAL, Location
SHOW TABLES DROPPED IN schema;  -- recently dropped managed tables
UNDROP TABLE schema.t;          -- restore within the retention period
```

## Setup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Sets the schema for this folder.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
No output.
</div>

In [0]:
CATALOG = "workspace"
DB = f"{CATALOG}.governance"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")

## 1. Create a managed table

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a table without a <code>LOCATION</code> and inspects its type and location.<br><br>
<b>Why it matters</b><br>Without <code>LOCATION</code>, Unity Catalog decides where the files go and manages their lifecycle. That's the default and the recommended choice.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>Type: MANAGED</code>. On Databricks the location is a path in the metastore's managed storage, which you never need to manage yourself.
</div>

In [0]:
spark.sql(f"CREATE OR REPLACE TABLE {DB}.orders_managed (id BIGINT, amount DOUBLE)")
spark.sql(f"INSERT INTO {DB}.orders_managed VALUES (1, 10.0), (2, 25.5), (3, 7.25)")

def table_info(name):
    rows = spark.sql(f"DESCRIBE TABLE EXTENDED {name}").filter("col_name IN ('Type', 'Location', 'Provider')").collect()
    return {r["col_name"]: r["data_type"] for r in rows}

print(table_info(f"{DB}.orders_managed"))

## 2. External tables: storage credentials and external locations

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Lists the external locations you can see, and shows how an external table would be created.<br><br>
<b>Why it matters</b><br>External tables need an external location that covers their path, and a privilege on it. Without one, <code>CREATE TABLE ... LOCATION</code> fails.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
Free Edition doesn't give you a cloud account, so you can't create storage credentials or external locations, and the list is probably empty or the command is denied. Read this section as a concept. On Azure Databricks (folder 09), you'll set up the real thing.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A list of external locations, or a message that none are available.
</div>

In [0]:
try:
    locations = spark.sql("SHOW EXTERNAL LOCATIONS").collect()
    print(f"{len(locations)} external location(s) visible")
    for r in locations[:10]:
        print(" ", r["name"], "->", r["url"])
except Exception as e:
    print("External locations not available here:", type(e).__name__)

example = """
-- Admin, once: credential + location
CREATE STORAGE CREDENTIAL landing_cred WITH (AZURE_MANAGED_IDENTITY = ...);   -- usually created in Catalog Explorer
CREATE EXTERNAL LOCATION landing URL 'abfss://landing@myacct.dfs.core.windows.net/'
  WITH (STORAGE CREDENTIAL landing_cred);
GRANT CREATE EXTERNAL TABLE, READ FILES ON EXTERNAL LOCATION landing TO `data-engineers`;

-- Engineer: external table
CREATE TABLE prod.raw.partner_orders (id BIGINT, amount DOUBLE)
LOCATION 'abfss://landing@myacct.dfs.core.windows.net/partner_orders';
"""
print(example)

## 3. What `DROP` does to files

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Simulates both behaviors with path-based Delta tables: a table registered over a path you own, dropped, and its files checked.<br><br>
<b>Why it matters</b><br>This is the difference you must remember: dropping an <b>external</b> table leaves the files. Dropping a <b>managed</b> table deletes them (after the retention period).
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
Locally, this demo uses a folder as the "external" path. On Databricks, the external path would be in an external location, which Free Edition doesn't have, so this cell uses a volume path only to show that the <b>files survive</b> independently of any table definition.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
After dropping the table registered over the path, the Delta files are still there and can still be read by path.
</div>

In [0]:
spark.sql(f"CREATE VOLUME IF NOT EXISTS {DB}.files")
BASE = "/Volumes/workspace/governance/files"

In [0]:
import os
path = f"{BASE}/external_orders"
spark.createDataFrame([(1, 10.0), (2, 25.5)], "id BIGINT, amount DOUBLE").write.format("delta").mode("overwrite").save(path)

try:
    spark.sql(f"CREATE TABLE IF NOT EXISTS {DB}.orders_external USING DELTA LOCATION '{path}'")
    print("registered:", table_info(f"{DB}.orders_external"))
    spark.sql(f"DROP TABLE {DB}.orders_external")
    print("table dropped")
except Exception as e:
    print("Can't register a table over this path here (expected on Free Edition: volumes aren't table locations):", type(e).__name__)

print("files still readable by path, rows:", spark.read.format("delta").load(path).count())

## 4. Drop and `UNDROP` a managed table

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Drops the managed table, lists dropped tables, and restores it with <code>UNDROP</code>.<br><br>
<b>Why it matters</b><br>Unity Catalog keeps dropped managed tables recoverable for 7 days. It's a safety net, not a backup strategy, but it saves many accidental drops.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
<code>UNDROP</code> is a Unity Catalog feature. If your workspace doesn't support it, the cell reports it and recreates the table.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The table appears in <code>SHOW TABLES DROPPED</code>, and after <code>UNDROP</code> it's back with its 3 rows.
</div>

In [0]:
spark.sql(f"DROP TABLE {DB}.orders_managed")
try:
    display(spark.sql(f"SHOW TABLES DROPPED IN {DB}"))
    spark.sql(f"UNDROP TABLE {DB}.orders_managed")
    print("restored rows:", spark.table(f"{DB}.orders_managed").count())
except Exception as e:
    print("UNDROP not available here:", type(e).__name__)
    spark.sql(f"CREATE OR REPLACE TABLE {DB}.orders_managed (id BIGINT, amount DOUBLE)")
    spark.sql(f"INSERT INTO {DB}.orders_managed VALUES (1, 10.0), (2, 25.5), (3, 7.25)")

## 5. Managed storage at catalog and schema level

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Shows how admins can choose where managed tables of a catalog or schema are stored, with <code>MANAGED LOCATION</code>.<br><br>
<b>Why it matters</b><br>Managed doesn't mean "one bucket for everything". Organizations often isolate data per catalog (for example a separate storage account for <code>prod_finance</code>) while keeping the benefits of managed tables.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The example DDL is printed. (Creating it requires an external location, so it isn't run.)
</div>

In [0]:
print("""
CREATE CATALOG prod_finance MANAGED LOCATION 'abfss://finance@myacct.dfs.core.windows.net/managed';
CREATE SCHEMA prod_finance.gold MANAGED LOCATION 'abfss://finance@myacct.dfs.core.windows.net/managed/gold';
-- managed tables in prod_finance.gold now live under that path, still fully managed by Unity Catalog
""")

## Under the hood

```
managed:   CREATE TABLE t (...)              ─▶ UC picks  <managed storage>/tables/<uuid>/
           DROP TABLE t                      ─▶ metadata removed, files deleted after retention (UNDROP ≤ 7 days)

external:  CREATE TABLE t (...) LOCATION p   ─▶ UC checks: p is inside an external location you can use
           DROP TABLE t                      ─▶ metadata removed, files at p untouched
```

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Metadata</span> the table entry in Unity Catalog points at the storage path; the Delta log at that path describes the data.

In [0]:
spark.sql(f"DESCRIBE DETAIL {DB}.orders_managed").select("format", "numFiles", "location").show(truncate=False)

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: external tables "for safety"</b><br>
You lose predictive optimization and <code>UNDROP</code>, and orphaned files pile up. Use managed tables unless other systems need direct access to the files.<br><br>
**⛔ Problem: <code>CREATE TABLE ... LOCATION</code> fails with a permission or "not in external location" error**<br>The path must be inside an external location, and you need <code>CREATE EXTERNAL TABLE</code> on it.<br><br>
**⛔ Problem: data still there after <code>DROP TABLE</code>**<br>It was an external table. Delete the files separately if you really mean to (carefully).<br><br>
<b>⛔ Problem: two tables pointing at the same path</b><br>Unity Catalog prevents overlapping paths for tables, which protects you from two definitions writing the same files.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is the difference between a managed and an external table?</b><br>
For a managed table, Unity Catalog chooses and manages the storage location and lifecycle: dropping the table deletes the data (recoverable with <code>UNDROP</code> for 7 days). For an external table, you provide a <code>LOCATION</code> inside an external location: dropping it removes only the metadata, and the files stay.<br><br>

<b>🎤 2. When would you use an external table?</b><br>
When other systems must read or write the files directly at a known path, when data must outlive the table definition, or for non-Delta formats in a landing zone. Otherwise managed tables are recommended.<br><br>

<b>🎤 3. What are storage credentials and external locations?</b><br>
A storage credential wraps a cloud identity (an Azure managed identity, an AWS IAM role, a GCP service account) that can access storage. An external location combines a credential with a storage path. Privileges on the external location control who can create external tables or read files there, so users never handle cloud keys.<br><br>

<b>🎤 4. What benefits do managed tables get?</b><br>
Predictive optimization (automatic <code>OPTIMIZE</code>, <code>VACUUM</code> and statistics), automatic liquid clustering, <code>UNDROP</code>, and faster metadata operations, all without managing paths.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A data engineer drops an external table in Unity Catalog. What happens?</b><br>
A. The metadata and data files are deleted<br>
B. Only the metadata is removed. The data files remain at the storage location<br>
C. The table is moved to <code>hive_metastore</code><br>
D. The command fails because external tables can't be dropped<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. Which object must exist before a user can create an external table at an <code>abfss://</code> path?</b><br>
A. A SQL warehouse<br>
B. An external location covering that path (with a storage credential)<br>
C. A Delta Sharing recipient<br>
D. A materialized view<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q3. What is the recommended default table type in Unity Catalog?</b><br>
A. External tables<br>
B. Managed tables<br>
C. Hive metastore tables<br>
D. Temporary views<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Create a managed table <code>practice_managed</code> with 2 rows and print its type</li><li>Write a Delta folder to a path (a volume on Databricks), read it back by path, then delete the folder with <code>dbutils.fs.rm</code> (Databricks) and confirm it's gone</li><li>Explain in a comment which of the two would survive a <code>DROP TABLE</code> and why</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Managed table
spark.sql(f"CREATE OR REPLACE TABLE {DB}.practice_managed (id INT, note STRING)")
spark.sql(f"INSERT INTO {DB}.practice_managed VALUES (1, 'a'), (2, 'b')")
print(table_info(f"{DB}.practice_managed")["Type"])

# 2. Delta folder at a path
p = f"{BASE}/practice_path_table"
spark.range(5).write.format("delta").mode("overwrite").save(p)
print("rows by path:", spark.read.format("delta").load(p).count())

# 3. A managed table's files are deleted on DROP (after retention); files at a path you own (external)
#    are untouched by DROP TABLE and must be removed separately.
spark.sql(f"DROP TABLE {DB}.practice_managed")

In [0]:
dbutils.fs.rm(p, recurse=True)
print("path removed:", not any(f.name.startswith("practice_path_table") for f in dbutils.fs.ls(BASE)))

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Managed: Unity Catalog owns the location and lifecycle. <code>DROP</code> deletes data (after retention). <code>UNDROP</code> within 7 days. The default choice</li><li>External: you give a <code>LOCATION</code> inside an external location. <code>DROP</code> removes metadata only</li><li>Storage credential (cloud identity) → external location (credential + path) → external tables and volumes</li><li><code>MANAGED LOCATION</code> on a catalog or schema chooses where its managed data lives</li><li>Check with <code>DESCRIBE TABLE EXTENDED</code> (Type, Location) or <code>DESCRIBE DETAIL</code></li></ul>
</div>

| Question | Answer |
|---|---|
| Who chooses the path? | Managed: UC. External: you (`LOCATION`) |
| `DROP TABLE` | Managed: data deleted. External: files kept |
| Recover a drop | `UNDROP TABLE` (managed, 7 days) |
| Needed for external | Storage credential + external location + privilege |
| Type check | `DESCRIBE TABLE EXTENDED t` |

## Git commit

```
git add 06_unity_catalog/02_managed_vs_external.ipynb
git commit -m "add 06_02 managed vs external notebook"
```